<a href="https://colab.research.google.com/github/abdullahh-sheikhh/voxelmorph/blob/dev/scripts/cell_tracking/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell Deformation Estimation with VoxelMorph

2D deformable registration for PhC-C2DH-U373 cells using VoxelMorph, compared against classical optical flow baselines.

## 1. Setup

In [ ]:
!git clone https://github.com/abdullahh-sheikhh/voxelmorph.git
%cd voxelmorph
!git checkout dev
!pip install -e . -q
!pip install imagecodecs -q

Cloning into 'voxelmorph'...
remote: Enumerating objects: 6081, done.
remote: Counting objects: 100% (823/823), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 6081 (delta 748), reused 690 (delta 672), pack-reused 5258 (from 3)
Receiving objects: 100% (6081/6081), 217.08 MiB | 19.54 MiB/s, done.
Resolving deltas: 100% (3558/3558), done.
/content/voxelmorph
Already on 'dev'
Your branch is up to date with 'origin/dev'.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Building editable for voxelmorph (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 51.9 MB/s eta 0:00:00


In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 2. Download Dataset

In [ ]:
import os
from pathlib import Path

# Download PhC-C2DH-U373 dataset from Cell Tracking Challenge
!mkdir -p dataset

if not Path('dataset/train/01').exists():
    print('Downloading training data...')
    !wget -q https://data.celltrackingchallenge.net/training-datasets/PhC-C2DH-U373.zip -O dataset/train.zip
    !cd dataset && unzip -q train.zip -d train_raw && mv train_raw/PhC-C2DH-U373/* train/ && rm -rf train_raw train.zip
    # Rename to match expected structure (CTC uses different naming)
    !ls dataset/train/
else:
    print('Training data already exists')

if not Path('dataset/test/01').exists():
    print('Downloading test data...')
    !wget -q https://data.celltrackingchallenge.net/test-datasets/PhC-C2DH-U373.zip -O dataset/test.zip
    !cd dataset && unzip -q test.zip -d test_raw && mv test_raw/PhC-C2DH-U373/* test/ && rm -rf test_raw test.zip
    !ls dataset/test/
else:
    print('Test data already exists')

# Verify
!echo "Train:" && ls dataset/train/01/ | head -5 && echo "..." && ls dataset/train/01/ | wc -l
!echo "Test:"  && ls dataset/test/01/  | head -5 && echo "..." && ls dataset/test/01/  | wc -l

Training data already exists
Test data already exists
Train:
t000.tif
t001.tif
t002.tif
t003.tif
t004.tif
...
115
Test:
t000.tif
t001.tif
t002.tif
t003.tif
t004.tif
...
115


## 3. Train

In [ ]:
# Train VM-1 (MSE, direct)
!python -m scripts.cell_tracking.train \n    --data-dir dataset/train \n    --epochs 100 \n    --batch-size 4 \n    --lr 1e-4 \n    --lambda 0.01 \n    --output-dir output \n    --save-every 25


## 4. Evaluate

In [ ]:
# Evaluate with baselines (Identity, Horn & Schunck, TV-L1) and Dice
!python -m scripts.cell_tracking.evaluate \
    --model output/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --output-dir output/eval \
    --max-pairs 0

In [ ]:
# Display evaluation and comparison visualizations
from IPython.display import Image, display
from pathlib import Path

eval_dir = Path('output/eval')

# Per-pair VoxelMorph analysis
for img_path in sorted(eval_dir.glob('pair_*.png'))[:3]:
    print(f'\n{img_path.name}')
    display(Image(filename=str(img_path), width=900))

# Baseline comparison (shared-scale SE maps)
for img_path in sorted(eval_dir.glob('comparison_*.png'))[:3]:
    print(f'\n{img_path.name}')
    display(Image(filename=str(img_path), width=900))

## 5. Model Variant Experiments

Train and compare different configurations from the paper (Table I):
- **VM-1**: MSE loss, direct displacement (baseline, already trained above)
- **VM-2**: NCC loss, direct displacement
- **VM-3**: MSE loss, diffeomorphic (int_steps=7)
- **VM-4**: NCC loss, diffeomorphic (int_steps=7)


In [ ]:
# VM-2: NCC loss, direct displacement
!python -m scripts.cell_tracking.train \n    --data-dir dataset/train \n    --epochs 100 --batch-size 4 --lr 1e-4 \n    --loss ncc --lambda 1.0 --int-steps 0 \n    --output-dir output/vm2_ncc \n    --save-every 25


In [ ]:
# VM-3: MSE loss, diffeomorphic (int_steps=7)
!python -m scripts.cell_tracking.train \n    --data-dir dataset/train \n    --epochs 100 --batch-size 4 --lr 1e-4 \n    --loss mse --lambda 0.01 --int-steps 7 \n    --output-dir output/vm3_diffeo \n    --save-every 25


In [ ]:
# VM-4: NCC loss, diffeomorphic (int_steps=7)
!python -m scripts.cell_tracking.train \n    --data-dir dataset/train \n    --epochs 100 --batch-size 4 --lr 1e-4 \n    --loss ncc --lambda 1.0 --int-steps 7 \n    --output-dir output/vm4_diffeo_ncc \n    --save-every 25


In [ ]:
# Evaluate all variants (baselines already computed in step 4)
from pathlib import Path

variants = {
    'VM-1 (MSE, direct)': ('output/best.pt', 0, 'output/eval_vm1'),
    'VM-2 (NCC, direct)': ('output/vm2_ncc/best.pt', 0, 'output/eval_vm2'),
    'VM-3 (MSE, diffeo)': ('output/vm3_diffeo/best.pt', 7, 'output/eval_vm3'),
    'VM-4 (NCC, diffeo)': ('output/vm4_diffeo_ncc/best.pt', 7, 'output/eval_vm4'),
}

for name, (model_path, int_steps, eval_dir) in variants.items():
    if not Path(model_path).exists():
        print(f'Skipping {name}: {model_path} not found')
        continue
    print(f'Evaluating {name}...')
    !python -m scripts.cell_tracking.evaluate \n        --model {model_path} \n        --data-dir dataset/train \n        --gt-dir dataset/train \n        --int-steps {int_steps} \n        --output-dir {eval_dir} \n        --no-baselines \n        --max-pairs 0


In [ ]:
# Comparison table: baselines + all VoxelMorph variants
import json
from IPython.display import display, Markdown
from pathlib import Path

rows = ['| Method | MSE | Dice | Folding % | Runtime (s/pair) |',
        '|--------|-----|------|-----------|------------------|']

bl_path = Path('output/eval') / 'metrics.json'
if bl_path.exists():
    with open(bl_path) as f:
        bm = json.load(f)
    for key, label in [('identity', 'Identity'), ('horn_schunck', 'Horn & Schunck'), ('tvl1', 'TV-L1')]:
        if key in bm:
            m = bm[key]
            dice = f"{m["dice_mean"]:.4f} +/- {m["dice_std"]:.4f}" if "dice_mean" in m else "N/A"
            rt = f"{m["runtime_mean"]:.4f}" if "runtime_mean" in m else "N/A"
            rows.append(f"| {label} | {m["mse_mean"]:.6f} +/- {m["mse_std"]:.6f} | {dice} | {m["folding_mean"]:.2f}% | {rt} |")

for name, edir in [
    ('VM-1 (MSE, direct)', 'output/eval_vm1'),
    ('VM-2 (NCC, direct)', 'output/eval_vm2'),
    ('VM-3 (MSE, diffeo)', 'output/eval_vm3'),
    ('VM-4 (NCC, diffeo)', 'output/eval_vm4'),
]:
    mp = Path(edir) / 'metrics.json'
    if not mp.exists():
        continue
    with open(mp) as f:
        data = json.load(f)
    m = data.get('vxm', data)
    dice = f"{m["dice_mean"]:.4f} +/- {m["dice_std"]:.4f}" if "dice_mean" in m else "N/A"
    rt = f"{m["runtime_mean"]:.4f}" if "runtime_mean" in m else "N/A"
    rows.append(f"| {name} | {m["mse_mean"]:.6f} +/- {m["mse_std"]:.6f} | {dice} | {m["folding_mean"]:.2f}% | {rt} |")

display(Markdown(chr(10).join(rows)))


## 6. Download Results


In [ ]:
# Download all results
from google.colab import files

!zip -r results.zip output/best.pt output/final.pt output/config.json \n    output/eval/ \n    output/vm2_ncc/ output/vm3_diffeo/ output/vm4_diffeo_ncc/ \n    output/eval_vm1/ output/eval_vm2/ output/eval_vm3/ output/eval_vm4/ \n    2>/dev/null; true
files.download('results.zip')
